# Step 10-B1: LLM chấm thứ hai (Qwen2.5-7B-Instruct, vLLM) trên 4 bộ hiện có

Lặp lại ablation 2×2 (prompt paper/code × xáo/cố định) và đo thiên lệch vị trí với một LLM mã nguồn mở, để xem phát hiện 3 có phải chỉ là đặc tính của gpt-4o-mini.

- **Accelerator:** GPU T4 ×2. **Internet:** On. **Secret:** `HF_TOKEN` (như Step 7).
- **Input:** `qudar-beir-data`, `qudar-runs`, `qudar-patches` (bản mới có `llm_score.py --backend vllm`). **Không** gắn `qudar-llm` hay `qudar-llm-ablation` vào notebook này: tên file cache trùng nhau.
- **Thời gian ước tính:** cài đặt 5 phút; 3 lần nạp mô hình × 2–3 phút; khoảng 24.600 prompt ở 8–12 prompt/giây → 35–50 phút. Ô DAT (tùy chọn) thêm khoảng 10 phút. Tổng khoảng 1–1,5 giờ.
- **Sau khi chạy:** Save Version → tạo dataset `qudar-llm-qwen` từ thư mục `llm_qwen/`.

In [ ]:
import glob, os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
DATA  = "/kaggle/input/datasets/karrychuong/qudar-beir-data/data"
PATCH = os.path.dirname(glob.glob("/kaggle/input/**/llm_score.py", recursive=True)[0])
RUNS  = os.path.dirname(os.path.dirname(glob.glob("/kaggle/input/**/fiqa_test/retrieval_result.json", recursive=True)[0]))
print("PATCH =", PATCH, "\nRUNS =", RUNS, os.listdir(RUNS))
assert "--backend" in open(f"{PATCH}/llm_score.py").read(), "qudar-patches chưa phải bản mới"
!pip install -q vllm pytrec-eval-terrier
!pip install -q --no-deps beir
!git clone -q https://github.com/kaist-dmlab/QuDAR.git /kaggle/working/QuDAR
# Không pip install QuDAR: chỉ cần thư mục src, tránh đụng phiên bản torch/transformers của vLLM
L = (f"python {PATCH}/llm_score.py --repo /kaggle/working/QuDAR --runs {RUNS} --data-root {DATA} "
     f"--out /kaggle/working/llm_qwen --backend vllm --model Qwen/Qwen2.5-7B-Instruct")

In [ ]:
# A. Ablation 2x2 đủ query trên FiQA, SciDocs, Climate-FEVER (4 chế độ, 1 lần nạp mô hình)
!{L} --mode paper code paper_fixed code_shuffle --names fiqa_test scidocs_test climate-fever_test

In [ ]:
# B. QuDAR-llm (chế độ paper) trên toàn bộ 7.405 query HotpotQA, cho bảng chính
!{L} --mode paper --names hotpotqa_test

In [ ]:
# C. Ablation trên 1.500 query đầu của HotpotQA như Step 6 (chế độ paper lấy từ cache của ô B)
!{L} --mode paper code paper_fixed code_shuffle --names hotpotqa_test --max-queries 1500

In [ ]:
# D. (Tùy chọn) DAT với Qwen chấm
!{L} --mode dat --names fiqa_test scidocs_test climate-fever_test hotpotqa_test

In [ ]:
# Tóm tắt để gửi lại: chỉ số + điểm trung bình theo vị trí hiển thị (cùng cách tính với step5_analysis.py)
import json, random, pandas as pd, numpy as np
from pathlib import Path
ev = pd.read_csv("/kaggle/working/llm_qwen/llm_eval.csv").drop_duplicates(["name", "method"], keep="last")
print(ev[["name", "method", "queries", "failed", "Recall@10", "nDCG@10"]].to_string(index=False))
rows = []
for f in sorted(Path("/kaggle/working/llm_qwen").glob("llm_*_*.jsonl")):
    mode = next((m for m in ("paper_fixed", "code_shuffle", "paper", "code") if f.name.startswith(f"llm_{m}_")), None)
    if mode is None:
        continue
    pos = {i: [] for i in range(1, 5)}
    for line in f.read_text().splitlines():
        r = json.loads(line)
        if not r["scores"]:
            continue
        order = ["ES", "OS", "ED", "OD"]
        if mode in ("paper", "code_shuffle"):
            order = ["OS", "OD", "ES", "ED"]
            random.Random(r["qid"]).shuffle(order)
        for i, s in enumerate(order, 1):
            pos[i].append(r["scores"][s])
    rows.append({"file": f.stem, **{f"pos {i}": round(float(np.mean(v)), 2) for i, v in pos.items()}})
print(pd.DataFrame(rows).to_string(index=False))